# MuterBandung Audit & Training Ledger

Notebook ini dipakai sebagai catatan eksekusi audit MuterBandung. Kode produksi tetap berada di folder `Scripts/`, sedangkan notebook ini menjadi tempat dokumentasi langkah, command, ringkasan hasil, dan bukti evaluasi.

Prinsip audit:
- deterministic engine harus stabil sebelum LLM masuk;
- ground truth menjadi alat ukur utama;
- LLM tidak boleh mengarang destinasi, harga, jarak, atau fasilitas;
- semua perubahan besar harus bisa diuji ulang dari notebook ini.


## 1. Setup Path dan Konfigurasi

Jalankan cell ini dari notebook untuk memastikan path project, endpoint API, dan file evaluasi sudah benar.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

API_URL = os.getenv("MUTERBANDUNG_API_URL", "http://127.0.0.1:5000/api/recommend")
GROUNDTRUTH_PATH = PROJECT_ROOT / "Dataset" / "3_Curated" / "groundtruth_queries.csv"
QC_REPORT_PATH = PROJECT_ROOT / "qc_report.md"
GT_REPORT_PATH = PROJECT_ROOT / "groundtruth_eval_report.md"
GT_JSON_PATH = PROJECT_ROOT / "groundtruth_eval_results.json"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("API_URL:", API_URL)
print("GROUNDTRUTH_PATH:", GROUNDTRUTH_PATH)


## 2. Baseline QC

Script `scratch_qc.py` adalah QC operasional cepat. Script ini sudah memiliki preflight sehingga tidak menimpa report valid saat API mati.

In [ ]:
result = subprocess.run(
    [sys.executable, "-B", "scratch_qc.py"],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
print("exit_code:", result.returncode)


In [ ]:
print(QC_REPORT_PATH.read_text(encoding="utf-8").split("## Detailed Results")[0])


## 3. Ground Truth Evaluator

Evaluator formal membaca `groundtruth_queries.csv`, memanggil API, lalu mengecek intent, kategori, label, no-strong-match, landmark, jarak, subtype belanja, flags fasilitas, crowd level, dan harga.

In [ ]:
result = subprocess.run(
    [sys.executable, "-B", "Scripts/evaluate_groundtruth.py"],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
print("exit_code:", result.returncode)


In [ ]:
summary = json.loads(GT_JSON_PATH.read_text(encoding="utf-8"))["summary"]
summary


## 4. Audit Matrix Sebelum LLM

Checklist tegas sebelum LLM boleh masuk ke sistem:

- [ ] QC operasional `scratch_qc.py` tetap PASS.
- [ ] Ground truth evaluator tetap PASS.
- [ ] Data lineage sentiment jelas: IndoBERT vs TF-IDF/SVM tidak boleh ambigu.
- [ ] Facility flags diverifikasi: toilet, mushola, parkir, wheelchair, pet friendly, safety, 24 jam.
- [ ] Coordinate audit selesai untuk destinasi meragukan.
- [ ] Subtype belanja dilengkapi: Mall, Factory Outlet, Oleh-Oleh, Pasar Wisata.
- [ ] LLM hanya menerima kandidat valid dari backend.
- [ ] Output LLM divalidasi backend sebelum tampil ke user.
- [ ] LLM tidak boleh membuat destinasi baru di luar kandidat.
- [ ] LLM tidak boleh mengarang harga, jarak, jam buka, atau fasilitas.


## 5. Pemeriksaan Artefak Audit

Cell ini mengecek file penting yang harus ada setelah audit berjalan.

In [ ]:
required_files = [
    PROJECT_ROOT / "scratch_qc.py",
    PROJECT_ROOT / "Scripts" / "evaluate_groundtruth.py",
    GROUNDTRUTH_PATH,
    QC_REPORT_PATH,
    GT_REPORT_PATH,
    GT_JSON_PATH,
    PROJECT_ROOT / "Dokumentasi_Sistem" / "LLM_UPGRADE_BLUEPRINT.md",
    PROJECT_ROOT / "Dokumentasi_Sistem" / "DATA_LINEAGE_REPORT.md",
    PROJECT_ROOT / "Dokumentasi_Sistem" / "BASELINE_EVALUATION_PIPELINE_AUDIT.md",
]

for path in required_files:
    print(f"{path.relative_to(PROJECT_ROOT)} :: {'OK' if path.exists() else 'MISSING'}")


## 6. Catatan Audit Saat Ini

Status baseline terakhir:

- `scratch_qc.py`: 62/62 PASS, 0 failed, 0 errors.
- `groundtruth_queries.csv`: 62 kasus, ID unik, schema valid.
- `evaluate_groundtruth.py`: 62/62 PASS, 100% pass rate.

Risiko utama berikutnya bukan pada logic utama recommender, tetapi pada kelengkapan data verified dan kejelasan lineage sentiment.

## 7. Audit Baseline Evaluation Pipeline

Audit tegas tahap sebelumnya sudah dicatat di `Dokumentasi_Sistem/BASELINE_EVALUATION_PIPELINE_AUDIT.md`.

Hasil audit:

- Status: **PASSED WITH CONTROLLED RISKS**.
- `scratch_qc.py`: preflight berjalan, report tidak rusak saat API down.
- `groundtruth_queries.csv`: 62 kasus, 22 kolom konsisten, ID unik.
- `evaluate_groundtruth.py`: 62/62 PASS, 100% pass rate.
- Failure mode API down: exit code 2, report lama tetap aman.

Keputusan audit: baseline evaluation pipeline siap menjadi pagar sebelum eksperimen LLM. Risiko yang masih harus diselesaikan berikutnya adalah data verified dan sentiment lineage.

## 8. Sentiment Lineage Audit

Audit sentiment lineage sudah dicatat di `Dokumentasi_Sistem/SENTIMENT_LINEAGE_AUDIT.md`.

Verdict:

- Status: **NEEDS CORRECTION BEFORE LLM / ACADEMIC CLAIM**.
- Source sentiment aktif yang terbukti: **TF-IDF + LinearSVC baseline**.
- IndoBERT status: artefak training/model tersedia, tetapi belum terbukti menjadi source aktif scoring runtime.
- Runtime recommender memakai `avg_sentimen_skor` dari CSV aktif.
- Field API `score_breakdown.sentimen_indobert` misleading karena nilainya bukan terbukti dari IndoBERT.

Keputusan audit: sebelum LLM masuk, backend perlu membawa metadata sentiment yang jujur seperti `sentiment_model_source`, `sentiment_model_version`, `sentiment_available`, dan `sentiment_score`.

## Audit Sentiment Metadata Netral - 2026-05-24

Tahap ini memperbaiki lineage sentiment tanpa mengubah ranking. Skor aktif tetap berasal dari `avg_sentimen_skor`, tetapi API dan dataset sekarang memakai field netral: `sentiment_score`, `sentiment_model_source`, `sentiment_model_version`, dan `sentiment_available`.

Keputusan audit: skor aktif tidak boleh disebut IndoBERT sampai pipeline IndoBERT benar-benar tersambung dan tervalidasi. Source aktif saat ini adalah `tfidf_linearsvc` dengan versi `run_nlp_pipeline_v2`.


In [ ]:
from pathlib import Path
import pandas as pd

paths = [
    Path("Dataset/3_Curated/DATABASE_WISATA_LABELED_V2_REVIEWED.csv"),
    Path("Dataset/3_Curated/DATABASE_WISATA_LABELED_V2.csv"),
    Path("DATABASE_WISATA_FINAL_PARIPURNA.csv"),
    Path("Dataset/SENTIMENT_SCORES_PER_LOKASI.csv"),
]
required = {
    "sentiment_score",
    "sentiment_model_source",
    "sentiment_model_version",
    "sentiment_available",
}

for path in paths:
    df = pd.read_csv(path, low_memory=False)
    missing = required.difference(df.columns)
    print(path, "rows=", len(df), "missing=", sorted(missing))
    if not missing:
        print(df[["sentiment_model_source", "sentiment_available"]].value_counts(dropna=False).head())

# Setelah server direstart, jalankan:
# python -B .\Scripts\evaluate_groundtruth.py
# python -B .\scratch_qc.py


## Hasil Validasi Sentiment Metadata - 2026-05-24

- AST parse semua file Python yang disentuh: OK.
- Unit test `python -B -m unittest Scripts.test_recommender`: 9 test OK.
- API contract via Flask test client: `sentiment_score` ada, source `tfidf_linearsvc`, version `run_nlp_pipeline_v2`, dan `sentimen_indobert` tidak keluar.
- Groundtruth evaluator via Flask test client: 62/62 PASS, 0 failed, 0 errors.
- QC via Flask test client: 62/62 PASS, final status PASSED WITH NOTES.
- Catatan operasional: server live port 5000 yang sudah berjalan sebelum perubahan masih perlu direstart agar field baru aktif di API live/browser.


## Final Live Audit - 2026-05-24

Audit final dijalankan terhadap server live http://127.0.0.1:5000/api/recommend.

Hasil utama:

- API live memakai kontrak sentiment baru: sentiment_score, sentiment_model_source=tfidf_linearsvc, sentiment_model_version=run_nlp_pipeline_v2, dan tidak lagi mengeluarkan sentimen_indobert.
- Probe stabilitas 20 request: 20/20 memakai kontrak baru.
- Dataset sentiment metadata complete; sentiment_score sama dengan vg_sentimen_skor untuk menjaga ranking tetap tidak berubah.
- Groundtruth evaluator live: 62/62 PASS, 0 failed, 0 errors.
- QC live: 62/62 PASS, final PASSED WITH NOTES.
- Unit test backend: 9 test OK.

Catatan operasional: port 5000 masih menunjukkan dua proses Python listening. Response sudah konsisten benar, tetapi sebelum demo sebaiknya hanya satu proses server yang aktif. Detail audit disimpan di Dokumentasi_Sistem/FINAL_LIVE_AUDIT_REPORT_2026-05-24.md.


## LLM Evidence Pack Implementation - 2026-05-24

Tahap ini membuat field llm_evidence_pack untuk response POST /api/recommend. Evidence pack mengambil kandidat valid dari backend dan membentuk data ringkas untuk LLM explanation layer.

File utama:

- Scripts/llm_evidence_pack.py
- Scripts/app.py
- Scripts/evaluate_groundtruth.py
- scratch_qc.py
- Scripts/test_llm_evidence_pack.py
- Dokumentasi_Sistem/LLM_EVIDENCE_PACK_SPEC.md

Guardrail utama: LLM hanya boleh menjelaskan kandidat dalam llowed_destination_ids, tidak boleh membuat destinasi baru, dan belum boleh mengubah ranking backend.

Status: valid via Flask test client. Server live perlu restart agar field baru muncul di API live.


## LLM Evidence Pack Live Validation - 2026-05-24

Evidence pack sudah divalidasi pada server live http://127.0.0.1:5000/api/recommend.

Hasil validasi:

- llm_evidence_pack.schema_version = muterbandung.llm_evidence_pack.v1
- llm_may_create_destinations = false
- llm_may_rerank = false
- Strict JSON serialization: OK
- Groundtruth live: 62/62 PASS, 0 failed, 0 errors
- QC live: 62/62 PASS, final PASSED WITH NOTES
- Unit test total: 12/12 OK
- Port 5000 hanya satu LISTENING process aktif setelah restart.

Status tahap: IMPLEMENTED AND LIVE-VALIDATED.


## LLM Prompt Guard + Output Validator

`MUTERBANDUNG_LLM_PROMPT_GUARD_VALIDATOR_2026_05_25`

Tahap ini menambahkan pagar produksi untuk lapisan LLM MuterBandung. LLM tetap hanya menjadi explanation layer: kandidat, ranking, harga, jarak, jam buka, sentiment, media URL, dan real-world flags harus berasal dari `llm_evidence_pack`.

Komponen yang ditambahkan:

- `Scripts/llm_guard.py`: pembuat `llm_prompt_guard` dan validator output LLM.
- `POST /api/recommend`: sekarang mengembalikan `llm_evidence_pack` dan `llm_prompt_guard`.
- `POST /api/llm/validate`: menolak output LLM yang mengarang destinasi, harga, jarak, URL gambar/link, website, rating, atau fasilitas.
- `candidate.media`: kontrak media aman. Jika URL gambar/link belum ada di curated dataset, `media.available=false`.

Catatan audit media:

- Pada saat guard dibuat, dataset curated aktif belum memiliki kolom URL gambar atau link destinasi.
- Raw Apify/Google Maps punya kandidat `imageUrl`, `url`, dan `website`.
- Status terbaru: media enrichment sudah dilakukan pada bagian pipeline media enrichment setelah guard ini.


In [ ]:
# Smoke test LLM guard dan validator output
# Jalankan dari root project PIJAK.

import sys
sys.path.insert(0, "Scripts")

from llm_evidence_pack import build_llm_evidence_pack
from llm_guard import build_llm_prompt_guard, validate_llm_output

sample_response = {
    "status": "success",
    "query": "wisata alam sejuk",
    "recommendations": [
        {
            "rank": 1,
            "location_id": "loc_001",
            "location_name": "Contoh Destinasi",
            "category": "Wisata Alam",
            "multi_labels": ["Alam"],
            "label_taxonomy": {"primary_intent": "Alam", "core_labels": ["Alam"]},
            "final_score": 88.5,
            "distance_km": None,
            "distance_label": None,
            "score_breakdown": {
                "sentiment_score": 0.91,
                "sentiment_model_source": "tfidf_linearsvc",
                "sentiment_model_version": "run_nlp_pipeline_v2",
                "sentiment_available": True,
            },
            "sentiment_metadata": {
                "sentiment_score": 0.91,
                "sentiment_model_source": "tfidf_linearsvc",
                "sentiment_model_version": "run_nlp_pipeline_v2",
                "sentiment_available": True,
            },
            "realworld_flags": {"coordinate_verified": True, "safety_verified": True},
            "info_praktis": {
                "harga": "Rp 15.000",
                "jam_buka_weekday": "08:00 - 17:00",
                "jam_buka_weekend": "08:00 - 17:00",
                "estimasi_durasi": "90 menit",
                "koordinat": [-6.9, 107.6],
            },
            "alasan": "Cocok dengan intent alam.",
        }
    ],
}

pack = build_llm_evidence_pack(sample_response)
guard = build_llm_prompt_guard(pack)
candidate = pack["candidates"][0]

llm_output = {
    "schema_version": "muterbandung.llm_output.v1",
    "answer": "Rekomendasi ini disusun sesuai evidence pack.",
    "selected_destination_ids": [candidate["destination_id"]],
    "destination_summaries": [
        {
            "destination_id": candidate["destination_id"],
            "rank": candidate["rank"],
            "name": candidate["name"],
            "why": candidate["backend_reason"],
            "price": candidate["practical_info"]["price"],
            "opening_hours": candidate["practical_info"]["opening_hours"],
            "distance_label": candidate["practical_info"]["distance_label"],
            "media": candidate["media"],
            "limitations": candidate["limitations"],
        }
    ],
    "warnings": [],
    "follow_up_question": None,
}

print("prompt_guard_schema:", guard["schema_version"])
print("valid_output:", validate_llm_output(llm_output, pack)["valid"])

llm_output["destination_summaries"][0]["price"] = "Rp 999.000"
print("fake_price_rejected:", not validate_llm_output(llm_output, pack)["valid"])


## Media Enrichment Pipeline + Groundtruth Audit

`MUTERBANDUNG_MEDIA_ENRICHMENT_PIPELINE_2026_05_25`

Tahap ini menambahkan URL gambar, link Google Maps, dan website ke dataset curated secara konservatif. Tujuannya agar frontend dan LLM evidence pack boleh menampilkan media hanya jika media tersebut benar-benar berasal dari raw data yang sudah cocok dengan `location_id`.

Kebijakan audit:

- Dataset aktif tetap `Dataset/3_Curated/DATABASE_WISATA_LABELED_V2_REVIEWED.csv`.
- Raw source: `Dataset/dataset_google-maps-extractor_2026-05-19_08-42-08-170.csv` dan `Dataset/1_Raw_Data/apify_jam_buka_semua_lokasi_raw.csv`.
- Match otomatis hanya menerima exact normalized title dan fuzzy high-confidence.
- Match borderline/mencurigakan tidak diaktifkan; masuk review queue.
- Road-level result seperti `Jl. Curug Panganten` dan `Jl. Gn. Tampomas` ditolak manual.
- `llm_guard` tetap menolak URL/fasilitas yang tidak ada di evidence pack.

Output audit:

- `Dokumentasi_Sistem/MEDIA_ENRICHMENT_AUDIT_2026-05-25.md`
- `Dataset/3_Curated/media_groundtruth_audit.csv`
- `Dataset/3_Curated/media_match_review_queue.csv`


In [ ]:
# Media enrichment smoke test
# Jalankan dari root project PIJAK.

import pandas as pd

curated_path = "Dataset/3_Curated/DATABASE_WISATA_LABELED_V2_REVIEWED.csv"
groundtruth_path = "Dataset/3_Curated/media_groundtruth_audit.csv"
review_queue_path = "Dataset/3_Curated/media_match_review_queue.csv"

df = pd.read_csv(curated_path)
gt = pd.read_csv(groundtruth_path)
rq = pd.read_csv(review_queue_path)

media_cols = [col for col in df.columns if col.startswith("media_")]
print("media_columns:", media_cols)
print("rows:", len(df))
print("accepted_media_rows:", int(df["media_available"].sum()))
print("missing_or_review_rows:", int((~df["media_available"]).sum()))
print("groundtruth_rows:", len(gt))
print("review_queue_rows:", len(rq))

sample = df[df["media_available"] == True][[
    "location_id",
    "location_name",
    "media_match_title",
    "media_match_method",
    "media_image_url",
    "media_destination_url",
]].head(5)
display(sample)


## Manual Completion Queues

`MUTERBANDUNG_MANUAL_COMPLETION_QUEUES_2026_05_25`

Tahap ini membuat CSV kerja manual untuk mengisi data yang masih kosong atau belum terverifikasi. CSV ini tidak langsung mengubah dataset utama; reviewer mengisi kolom `new_*`, `new_value`, atau kolom reviewer terlebih dahulu, lalu perubahan nanti diterapkan lewat pipeline apply/merge terpisah agar tetap auditable.

Output:

- `Dataset/3_Curated/manual_media_fill_queue.csv`
- `Dataset/3_Curated/manual_data_fill_queue.csv`
- `Dataset/3_Curated/manual_realworld_verification_queue.csv`
- `Dokumentasi_Sistem/MANUAL_COMPLETION_QUEUES_AUDIT_2026-05-25.md`

Ringkasan awal:

- Media kosong/belum aman: 53 baris.
- Data non-media yang perlu isi/review: 60 isu.
- Real-world flags yang perlu verifikasi: 213 destinasi aktif.


In [ ]:
# Audit ringkas manual completion queues
import pandas as pd

media_queue = pd.read_csv("Dataset/3_Curated/manual_media_fill_queue.csv")
data_queue = pd.read_csv("Dataset/3_Curated/manual_data_fill_queue.csv")
realworld_queue = pd.read_csv("Dataset/3_Curated/manual_realworld_verification_queue.csv")

print("media_missing_rows:", len(media_queue))
print("data_issue_rows:", len(data_queue))
print("realworld_verification_rows:", len(realworld_queue))
print("\nMedia priority:")
print(media_queue["priority"].value_counts().to_string())
print("\nData issue type:")
print(data_queue["issue_type"].value_counts().to_string())
display(media_queue.head(10))
display(data_queue.head(10))


## Media Fill Google Colab Workflow

`MUTERBANDUNG_MEDIA_FILL_GOOGLE_COLAB_2026_05_25`

Tahap ini menyiapkan workflow Google Colab untuk pengisian media saja, tanpa Apify. Colab digunakan untuk membuka link bantu Google Search/Maps/Images, mengisi URL gambar/link destinasi, memvalidasi URL, lalu mengekspor CSV apply-ready.

File penting:

- `Notebooks/Media_Fill_Google_Colab.ipynb`
- `Dataset/3_Curated/manual_media_fill_queue.csv`
- `Scripts/apply_manual_media_updates.py`
- `Dokumentasi_Sistem/MEDIA_FILL_COLAB_WORKFLOW.md`

Aturan:

- Colab tidak langsung mengubah dataset utama.
- Isi `reviewer_status=approved` untuk baris yang siap dipakai.
- Download `media_manual_apply_ready.csv`, lalu apply lokal dengan script backup-safe.


In [ ]:
# Ringkasan workflow Colab media fill
import pandas as pd

queue = pd.read_csv("Dataset/3_Curated/manual_media_fill_queue.csv")

print("media queue rows:", len(queue))
print("high priority:", int((queue["priority"] == "HIGH").sum()))
print("has helper search columns:", all(col in queue.columns for col in [
    "google_search_url",
    "google_maps_search_url",
    "google_image_search_url",
]))
display(queue.head(5))


## Colab Research Notebooks

`MUTERBANDUNG_COLAB_RESEARCH_NOTEBOOKS_2026_05_25`

Tahap ini menyiapkan notebook Google Colab untuk membantu pencarian data manual tanpa Apify. Colab dipakai untuk membuka link bantu, mengisi nilai hasil riset, memvalidasi input, lalu mengekspor file `apply_ready`.

Notebook yang disiapkan:

- `Notebooks/Media_Fill_Google_Colab.ipynb`
- `Notebooks/Data_Completion_Google_Colab.ipynb`
- `Notebooks/Realworld_Verification_Google_Colab.ipynb`

Prinsip:

- Colab tidak menulis dataset utama.
- Baris harus `reviewer_status=approved` agar masuk apply-ready.
- Dataset utama baru diubah setelah file hasil Colab diaudit lokal.


In [ ]:
import pandas as pd

queues = {
    "media": "Dataset/3_Curated/manual_media_fill_queue.csv",
    "data": "Dataset/3_Curated/manual_data_fill_queue.csv",
    "realworld": "Dataset/3_Curated/manual_realworld_verification_queue.csv",
}

for name, path in queues.items():
    queue = pd.read_csv(path)
    print(name, "rows:", len(queue), "columns:", len(queue.columns))


## Local Scrape Candidates

`MUTERBANDUNG_LOCAL_SCRAPE_CANDIDATES_2026_05_25`

Tahap ini membuat kandidat pengisian data dari raw Google Maps extractor lokal, tanpa Colab dan tanpa mengubah dataset utama.

Output:

- `Dataset/3_Curated/local_scrape_media_candidates.csv`
- `Dataset/3_Curated/local_scrape_data_candidates.csv`
- `Dataset/3_Curated/local_scrape_unresolved_queue.csv`
- `Dokumentasi_Sistem/LOCAL_SCRAPE_AUDIT_2026-05-25.md`

Ringkasan:

```json
{
  "raw_rows": 230,
  "media_queue_rows": 53,
  "data_queue_rows": 60,
  "media_candidates": 25,
  "data_candidates": 10,
  "unresolved": 78
}
```


In [ ]:
import pandas as pd

for path in [
    "Dataset/3_Curated/local_scrape_media_candidates.csv",
    "Dataset/3_Curated/local_scrape_data_candidates.csv",
    "Dataset/3_Curated/local_scrape_unresolved_queue.csv",
]:
    df = pd.read_csv(path)
    print(path, "rows:", len(df))
    display(df.head(5))


# Hotel canonical dataset foundation (2026-05-29)

Bagian ini menyimpan pekerjaan pondasi hotel sebelum masuk ke LLM atau model rekomendasi. Dataset hotel tidak dibuang walaupun berisi hotel, guest house, villa, apartment, vacation rental, dan listing level kamar. Semua dipertahankan, tetapi diberi `property_segment` agar recommender bisa memfilter sesuai kebutuhan user.

Output utama:
- `Dataset/3_Curated/HOTEL_CANONICAL_CIMAHI_2026-05-29.csv`
- `Dataset/3_Curated/hotel_canonical_summary_2026-05-29.json`
- `Dataset/3_Curated/hotel_canonical_validation_results_2026-05-29.json`
- `Dokumentasi_Sistem/HOTEL_CANONICAL_PIPELINE_2026-05-29.md`
- `Dokumentasi_Sistem/HOTEL_VALIDATION_REPORT_2026-05-29.md`

Status terakhir:
- Baris: 181
- Kolom: 76
- Validation gate: PASS_WITH_WARNINGS
- Error: 0
- Warning: 469
- Segment properti: apartment: 65, hotel: 64, guest_house: 36, villa: 10, room_level_listing: 5, vacation_rental: 1
- Quality flags terisi: rating_available: 128, price_available: 89, amenities_available: 121, image_available: 180, description_available: 24, source_link_available: 94, reviews_breakdown_available: 18
- Review confidence: missing_review_confidence: 53, low_review_confidence: 53, medium_review_confidence: 48, high_review_confidence: 27

Catatan penting: `rating_sentiment_score` dan `adjusted_rating_sentiment_score` adalah rating-based sentiment, bukan NLP komentar. Ini dipakai karena dataset hotel saat ini belum punya raw review comment yang cukup untuk pipeline NLP seperti wisata.


In [ ]:
# Rebuild canonical hotel dataset dan jalankan validation pipeline.
# Jalankan sel ini dari root project PIJAK atau dari folder Notebooks.
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "Scripts").exists() and (PROJECT_ROOT.parent / "Scripts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "Scripts" / "build_hotel_canonical_dataset.py")],
    check=True,
)
subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "Scripts" / "validate_hotel_canonical_dataset.py")],
    check=True,
)


In [ ]:
# Ringkasan cepat output hotel canonical.
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "Dataset").exists() and (PROJECT_ROOT.parent / "Dataset").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

canonical_path = PROJECT_ROOT / "Dataset" / "3_Curated" / "HOTEL_CANONICAL_CIMAHI_2026-05-29.csv"
summary_path = PROJECT_ROOT / "Dataset" / "3_Curated" / "hotel_canonical_summary_2026-05-29.json"
validation_path = PROJECT_ROOT / "Dataset" / "3_Curated" / "hotel_canonical_validation_results_2026-05-29.json"

hotel_df = pd.read_csv(canonical_path)
summary = json.loads(summary_path.read_text(encoding="utf-8"))
validation = json.loads(validation_path.read_text(encoding="utf-8"))

display(
    {
        "rows": summary["row_count"],
        "columns": summary["column_count"],
        "gate_status": validation["summary"]["gate_status"],
        "errors": validation["summary"]["error_count"],
        "warnings": validation["summary"]["warning_count"],
        "property_segment_counts": summary["property_segment_counts"],
        "review_confidence_counts": summary["review_confidence_counts"],
        "sentiment_source_counts": summary["sentiment_source_counts"],
    }
)

hotel_df[
    [
        "hotel_id",
        "name",
        "property_segment",
        "overall_rating",
        "reviews",
        "price_lowest",
        "rating_sentiment_score",
        "adjusted_rating_sentiment_score",
        "rating_sentiment_label",
        "review_confidence_label",
        "data_quality_score",
    ]
].head(10)


## Aturan penggunaan untuk recommender dan LLM

- Untuk user yang minta hotel formal, prioritaskan `property_segment == "hotel"` lalu fallback ke `guest_house` jika perlu.
- Untuk user yang minta keluarga, ramai-ramai, murah, atau staycation fleksibel, `villa`, `apartment`, dan `vacation_rental` boleh ikut.
- Baris dengan `missing_review_confidence` atau `low_review_confidence` tidak boleh dijelaskan terlalu percaya diri.
- Baris tanpa `price_available` tidak boleh dipakai untuk klaim murah/mahal secara tegas.
- Baris tanpa `amenities_available` tidak boleh membuat klaim fasilitas spesifik.
- Baris tanpa `image_available` jangan dipakai untuk kartu visual utama.
